# Дообучение RuModernBERT-base для классификации спам-сообщений

Ноутбук выполняет дообучение модели `deepvk/RuModernBERT-base` (150M параметров, ModernBERT архитектура 2025) на собранном датасете.

Особенности обучения:
- **FP16** — смешанная точность для ускорения обучения на GPU
- **FocalLoss** — функция потерь, фокусирующаяся на сложных примерах
- **EarlyStopping** — ранняя остановка при отсутствии улучшения F1

Обучение выполняется на трёх вариантах текста:
- исходный текст (raw)
- нормализованный текст (normalized)
- предобработанный текст (preprocessed)

Фильтрация: остаются только сообщения, содержащие преимущественно кириллицу.

## Импорт необходимых библиотек

In [1]:
import os
import sys

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
    pipeline,
)
from datasets import Dataset

try:
    _project_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
except NameError:
    _cwd = os.getcwd()
    _project_root = _cwd
    while _project_root != '/' and not os.path.isdir(os.path.join(_project_root, 'src')):
        _project_root = os.path.dirname(_project_root)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from src.models.transformer import (
    FocalLossTrainer,
    tokenize_function,
    compute_metrics,
    is_mostly_cyrillic,
    get_training_args,
    get_model_config,
    prepare_text_variants,
    benchmark_cpu_inference,
)
from src.config import PROCESSED_DIR, MODELS_DIR

## Чтение обработанного датасета

Загружается `preprocessed.csv` из `data/processed/`.

In [2]:
df = pd.read_csv(PROCESSED_DIR / 'preprocessed.csv', index_col=0)

## Подготовка вариантов текста

Из датасета выделяются три варианта текста для дообучения:
- **raw** — исходный текст (колонка `text`)
- **normalized** — Unicode-нормализация и удаление HTML-тегов
- **preprocessed** — полная предобработка (колонка `text_preprocessed`)

Фильтрация по кириллице: остаются только сообщения, где доля кириллических символов превышает 30%.

In [3]:
variants = prepare_text_variants(df)

ru_variants = {}
for name, vdf in variants.items():
    mask = vdf['text'].apply(lambda t: is_mostly_cyrillic(str(t)))
    ru_variants[name] = vdf[mask].reset_index(drop=True)
    print(f'{name}: {len(ru_variants[name])} сообщений')

raw: 72342 сообщений
normalized: 72355 сообщений
preprocessed: 72503 сообщений


## Дообучение RuModernBERT-base

Модель: `deepvk/RuModernBERT-base`.
Количество классов: 2 (ham=0, spam=1).

### Загрузка модели и токенайзера

In [4]:
MODEL_NAME = 'deepvk/RuModernBERT-base'
model_config = get_model_config(MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

### Подготовка датасетов и токенизация

Для каждого варианта текста создаётся отдельный train/test сплит (80/20) и токенизируется.

In [5]:
tokenized = {}
for name, vdf in ru_variants.items():
    ds = Dataset.from_pandas(vdf)
    split = ds.train_test_split(test_size=0.2, seed=42)
    train_ds = split['train']
    test_ds = split['test']

    train_tok = train_ds.map(
        tokenize_function,
        batched=True,
        fn_kwargs={'tokenizer': tokenizer, 'max_length': model_config['max_length']},
    )
    test_tok = test_ds.map(
        tokenize_function,
        batched=True,
        fn_kwargs={'tokenizer': tokenizer, 'max_length': model_config['max_length']},
    )

    tokenized[name] = {'train': train_tok, 'test': test_tok}
    print(f'{name}: train={{len(train_tok)}}, test={{len(test_tok)}}')

Map:   0%|          | 0/57873 [00:00<?, ? examples/s]

Map:   0%|          | 0/14469 [00:00<?, ? examples/s]

raw: train={len(train_tok)}, test={len(test_tok)}


Map:   0%|          | 0/57884 [00:00<?, ? examples/s]

Map:   0%|          | 0/14471 [00:00<?, ? examples/s]

normalized: train={len(train_tok)}, test={len(test_tok)}


Map:   0%|          | 0/58002 [00:00<?, ? examples/s]

Map:   0%|          | 0/14501 [00:00<?, ? examples/s]

preprocessed: train={len(train_tok)}, test={len(test_tok)}


### Создание trainer'ов

Используется `FocalLossTrainer`.
Для каждого варианта текста создаётся отдельный trainer.

In [6]:
model_cfg = {
    'learning_rate': 2e-5,
    'epochs': 5,
    'batch_size': 16,
    'fp16': True,
    'focal_alpha': 0.25,
    'focal_gamma': 2.0,
}

trainers = {}
for name in tokenized:
    output_dir = f'finetuned_rumodernbert_{name}'

    training_args = get_training_args(
        output_dir=output_dir,
        learning_rate=model_cfg['learning_rate'],
        num_train_epochs=model_cfg['epochs'],
        per_device_train_batch_size=model_cfg['batch_size'],
        per_device_eval_batch_size=model_cfg['batch_size'],
        max_length=model_config['max_length'],
        fp16=model_cfg['fp16'],
        gradient_checkpointing=model_config['gradient_checkpointing'],
    )

    trainer = FocalLossTrainer(
        focal_alpha=model_cfg['focal_alpha'],
        focal_gamma=model_cfg['focal_gamma'],
        model=model,
        args=training_args,
        train_dataset=tokenized[name]['train'],
        eval_dataset=tokenized[name]['test'],
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainers[name] = trainer

### Запуск дообучения

Обучение выполняется на GPU (если доступно).

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')
model.to(device)

Устройство: cuda


ModernBertForSequenceClassification(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=False)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=False)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernB

#### На исходных текстах (raw)

In [8]:
trainers['raw'].train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.001314,0.996337,0.990710,0.998587,0.982957
2,No log,0.002103,0.994264,0.985697,0.976776,0.994783
3,No log,0.001925,0.996890,0.992140,0.996491,0.987826
4,No log,0.002153,0.997097,0.992660,0.997541,0.987826
5,No log,0.001986,0.997443,0.993539,0.997546,0.989565


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18090, training_loss=0.00564209161508538, metrics={'train_runtime': 6490.0885, 'train_samples_per_second': 44.586, 'train_steps_per_second': 2.787, 'total_flos': 9.860339411241984e+16, 'train_loss': 0.00564209161508538, 'epoch': 5.0})

In [9]:
trainers['raw'].evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.001986,5,0.997443,0.993539,0.997546,0.989565


{'eval_loss': 0.001986365532502532,
 'eval_accuracy': 0.9974428087635635,
 'eval_f1': 0.9935393748908679,
 'eval_precision': 0.9975455820476858,
 'eval_recall': 0.9895652173913043}

Сохранение модели.

In [10]:
model.save_pretrained(MODELS_DIR / 'finetuned_rumodernbert_raw')
tokenizer.save_pretrained(MODELS_DIR / 'finetuned_rumodernbert_raw')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rumodernbert_raw/tokenizer_config.json',
 '/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rumodernbert_raw/tokenizer.json')

#### На нормализованных текстах (normalized)

In [11]:
trainers['normalized'].train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.000603,0.998134,0.995197,0.992900,0.997504
2,No log,0.000784,0.998065,0.994987,0.998922,0.991084
3,No log,0.000746,0.998549,0.996252,0.997142,0.995364
4,No log,0.000939,0.998825,0.996964,0.998569,0.995364
5,No log,0.000907,0.998894,0.997143,0.998569,0.995720


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18090, training_loss=0.00024422998254374287, metrics={'train_runtime': 6486.6069, 'train_samples_per_second': 44.618, 'train_steps_per_second': 2.789, 'total_flos': 9.862213579395072e+16, 'train_loss': 0.00024422998254374287, 'epoch': 5.0})

In [12]:
trainers['normalized'].evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.000907,5,0.998894,0.997143,0.998569,0.995720


{'eval_loss': 0.0009069226216524839,
 'eval_accuracy': 0.9988943404049478,
 'eval_f1': 0.9971428571428571,
 'eval_precision': 0.9985693848354793,
 'eval_recall': 0.9957203994293866}

Сохранение модели.

In [13]:
model.save_pretrained(MODELS_DIR / 'finetuned_rumodernbert_norm')
tokenizer.save_pretrained(MODELS_DIR / 'finetuned_rumodernbert_norm')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rumodernbert_norm/tokenizer_config.json',
 '/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rumodernbert_norm/tokenizer.json')

#### На предобработанных текстах (preprocessed)

In [8]:
trainers['preprocessed'].train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.002774,0.993862,0.984405,0.980796,0.988041
2,No log,0.001311,0.995449,0.988384,0.989081,0.987689
3,No log,0.001870,0.996000,0.989753,0.994320,0.985227
4,No log,0.002593,0.996000,0.989753,0.994320,0.985227
5,No log,0.002780,0.996000,0.989753,0.994320,0.985227


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18130, training_loss=0.0016215062969900005, metrics={'train_runtime': 6494.5043, 'train_samples_per_second': 44.655, 'train_steps_per_second': 2.792, 'total_flos': 9.882318292310016e+16, 'train_loss': 0.0016215062969900005, 'epoch': 5.0})

In [9]:
trainers['preprocessed'].evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.001870,5,0.996000,0.989753,0.994320,0.985227


{'eval_loss': 0.0018702750094234943,
 'eval_accuracy': 0.9960002758430453,
 'eval_f1': 0.9897526501766785,
 'eval_precision': 0.9943201987930422,
 'eval_recall': 0.9852268730214562}

Сохранение модели.

In [10]:
model.save_pretrained(MODELS_DIR / 'finetuned_rumodernbert_p')
tokenizer.save_pretrained(MODELS_DIR / 'finetuned_rumodernbert_p')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rumodernbert_p/tokenizer_config.json',
 '/home/sophrosyne/STANKIN_AntiSpam_Bot/models/finetuned_rumodernbert_p/tokenizer.json')

## Замер CPU-инференса

Замеряется latency инференса на CPU для оценки пригодности модели к развёртыванию на сервере без GPU.

In [11]:
test_messages = [
    "Это честное сообщение от пользователя.",
    "🔥 Казино онлайн! Зарабатывай миллионы прямо сейчас! 💰💎",
    "Зарабатывай миллионы **онлайн** прямо сейчас!",
    "Работа на дому, легкий доход. Пиши в личку!",
    "Привет! Как дела? У меня всё отлично.",
    "steam gift 50$ - steamcommunity.com/gift-card/pay/50\n@everyone",
    "Давайте **вместе** будем писать про казино в чатах!!! Присоединяйтесь!",
    "Как же надоели эти сообщения про казино",
    "Добрый день. Для подачи документов необходимо пройти регистрацию здесь: stankin.ru",
    "3-4 часа и 8 тысяч твои!  Пиши  https://t.me/rasmuswork1",
]

sample_texts = test_messages * 10
suffix_map = {'raw': 'raw', 'normalized': 'norm', 'preprocessed': 'p'}
cpu_results = {}
for name, suffix in suffix_map.items():
    model_path = str(MODELS_DIR / f'finetuned_rumodernbert_{suffix}')
    m = AutoModelForSequenceClassification.from_pretrained(model_path)
    tok = AutoTokenizer.from_pretrained(model_path)
    cpu_results[name] = benchmark_cpu_inference(m, tok, sample_texts, max_length=model_config['max_length'])
    print(f'{name}: {cpu_results[name]}')

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

raw: {'avg_ms_per_msg': 18.89, 'p95_ms_per_msg': 33.72, 'throughput_msgs_per_sec': 52.95}


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

normalized: {'avg_ms_per_msg': 20.4, 'p95_ms_per_msg': 39.57, 'throughput_msgs_per_sec': 49.03}


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

preprocessed: {'avg_ms_per_msg': 16.67, 'p95_ms_per_msg': 16.46, 'throughput_msgs_per_sec': 60.0}
